# Experiment 3 — MovieLens: real rating distributions

Figures per `docs/experiments.md`, Experiment 3 "Figures" subsection. This notebook only loads `results/exp3/results.parquet` and plots — it never recomputes algorithm results (see CLAUDE.md and `docs/experiments.md` §0.4). Run `make exp3` (or `make smoke`) first if that file is missing.

Reconstructing `MovieLensEnv` below (for the prototype titles/histograms in Figure 1) is *not* a recompute of algorithm results — prototype selection is a deterministic, scripted function of `data/ml-1m/` (`docs/experiments.md`: "notebooks/03 shows the result"), independent of any algorithm run.

**Note:** as of this run, `results/exp3/` holds the REAL (non-smoke) result: K=4 prototypes, 10 arms/prototype (N=40) for the main instance, plus the N=6 reduced instance for tas-fw, at full rep counts (100, 50 for tas-fw).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kabc.plotting import set_style, style_for, legend_for_algorithms, savefig, ALGO_ORDER, ALGO_COLORS, ALGO_LABELS
from kabc.environments import MovieLensEnv

set_style()

RESULTS_PATH = Path("../results/exp3/results.parquet")
if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"{RESULTS_PATH} not found. Run `make exp3` (or `make smoke`) to generate it — "
        "this notebook only plots existing results, it never recomputes them."
    )
df = pd.read_parquet(RESULTS_PATH)
print(f"loaded {len(df)} rows from {RESULTS_PATH}")

main_df = df[df["family"] == "movielens"].copy()
reduced_df = df[df["family"] == "movielens_reduced"].copy()
df.head()

loaded 550 rows from ../results/exp3/results.parquet


,exp_id,family,algorithm,rep,seed,swept_param,swept_value,budget_cap,budget_multiplier,s_star,...,Delta,sigma,sigma_B,a,ring_radius,ring_noise,data_dir,arms_per_prototype,min_ratings,mean_gap_tol
0,3,movielens,kabc,0,3000,None,None,2000000,1.0,None,...,None,None,None,None,None,None,data/ml-1m,10,1000,0.05
1,3,movielens,kabc,1,3001,None,None,2000000,1.0,None,...,None,None,None,None,None,None,data/ml-1m,10,1000,0.05
2,3,movielens,kabc,2,3002,None,None,2000000,1.0,None,...,None,None,None,None,None,None,data/ml-1m,10,1000,0.05
3,3,movielens,kabc,3,3003,None,None,2000000,1.0,None,...,None,None,None,None,None,None,data/ml-1m,10,1000,0.05
4,3,movielens,kabc,4,3004,None,None,2000000,1.0,None,...,None,None,None,None,None,None,data/ml-1m,10,1000,0.05


## Figure 1 (headline): all four prototypes' rating distributions + final ARI

Left: rating distribution (grouped bar chart, one thin bar per movie per
rating value 1-5, non-overlapping) for all K=4 prototypes selected by
`docs/experiments.md` Experiment 3 steps 1-3, reconstructed deterministically
from `data/ml-1m/` via `MovieLensEnv` — not an algorithm run. Two of these
four (annotated) are the polarizing-vs-consensus same-mean pair (the one
with `|mean gap| <= mean_gap_tol` maximizing MMD²) that motivates this
experiment; the other two separate the clusters further on ordinary
mean-based grounds. Right: final ARI per algorithm from the stored results,
annotated with mean τ.

In [2]:
# Reconstruct the deterministic prototype selection using the same base
# kwargs as configs/exp3.yaml's `movielens` family, to identify and plot the
# same-mean pair -- no algorithm/bandit logic is run here.
env = MovieLensEnv(
    data_dir="../data/ml-1m", K=4, arms_per_prototype=10, min_ratings=1000, mean_gap_tol=0.05
)
means = [pool.mean() for pool in env.prototype_pools]
pairs = [(i, j) for i in range(len(means)) for j in range(i + 1, len(means))]
same_mean_pair = min(pairs, key=lambda p: abs(means[p[0]] - means[p[1]]))
i1, i2 = same_mean_pair
print(f"same-mean pair: {env.prototype_titles[i1]!r} (mean={means[i1]:.2f}) vs "
      f"{env.prototype_titles[i2]!r} (mean={means[i2]:.2f})")

same-mean pair: 'Clear and Present Danger (1994)' (mean=3.73) vs 'South Park: Bigger, Longer and Uncut (1999)' (mean=3.76)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

ax = axes[0]
# Grouped bar chart, all 4 prototypes: one thin, non-overlapping bar per
# movie per rating value, rather than overlapping semi-transparent
# histograms (which only worked visually for 2 movies -- with 4, overlap
# would make individual distributions unreadable). Colorblind-safe
# qualitative palette, distinct from ALGO_COLORS (these are movies, not
# algorithms).
movie_colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]
ratings_support = np.array([1, 2, 3, 4, 5])
n_movies = len(env.prototype_titles)
bar_width = 0.8 / n_movies
for m, (title, pool, color) in enumerate(zip(env.prototype_titles, env.prototype_pools, movie_colors)):
    counts = np.array([(pool == r).mean() for r in ratings_support])  # density over {1..5}
    offset = (m - (n_movies - 1) / 2) * bar_width
    ax.bar(ratings_support + offset, counts, width=bar_width * 0.95, color=color,
           label=f"{title} (mean={pool.mean():.2f})")
ax.set_xticks(ratings_support)
ax.set_xlabel("rating")
ax.set_ylabel("density")
same_mean_short = f"{env.prototype_titles[i1].split('(')[0].strip()} & {env.prototype_titles[i2].split('(')[0].strip()}"
ax.set_title(f"all {n_movies} prototypes\nsame-mean pair: {same_mean_short}", fontsize=8)
ax.legend(fontsize=5.2, frameon=False, loc="upper left")

ax = axes[1]
# A bar per algorithm's final ARI, not a line "vs budget": the stored
# results only have each run's FINAL ari/tau, no intermediate checkpoints,
# so an earlier version's axhline (one full-width horizontal line per
# algorithm) wasn't really showing a trajectory either -- and when two
# algorithms land on the same ARI (e.g. both 0 at smoke scale), the lines
# exactly overlap and one becomes invisible. A bar chart stays legible
# regardless of whether values coincide, and is honest about what this
# data actually is: one endpoint value per algorithm, not a curve.
#
# All FOUR algorithms that actually ran on this main instance (matching
# Table 3's roster: kabc, kabc-sg, acb, unif-kkm -- see configs/exp3.yaml's
# `movielens` family) are shown here, not just acb/kabc -- an earlier
# version of this cell hardcoded ["acb", "kabc"], silently dropping
# kabc-sg/unif-kkm from the plot even though their rows exist in main_df.
algos_present = [a for a in ALGO_ORDER if a in main_df["algorithm"].unique()]
mean_aris = [main_df[main_df["algorithm"] == a]["ari"].mean() for a in algos_present]
colors = [ALGO_COLORS[a] for a in algos_present]
bars = ax.bar([ALGO_LABELS[a] for a in algos_present], mean_aris, color=colors, alpha=0.8, width=0.6)
for bar, a, val in zip(bars, algos_present, mean_aris):
    tau = main_df[main_df["algorithm"] == a]["tau"].mean()
    ax.annotate(f"ARI={val:.2f}\n$\\tau$={tau:.0f}", (bar.get_x() + bar.get_width() / 2, val),
                 ha="center", va="bottom" if val < 0.9 else "top", fontsize=6.5,
                 color="black" if val < 0.9 else "white")
ax.axhline(1.0, color="grey", linestyle=":", linewidth=1)
ax.set_ylabel("mean ARI (final)")
ax.set_ylim(-0.05, 1.15)
ax.set_title("final ARI, all algorithms (main instance)")
ax.tick_params(axis="x", labelsize=6.5)

fig.tight_layout()
savefig(fig, "exp3_fig1_same_mean_pair_and_ari", fig_dir="../results/figures")
plt.show()

## Figure 2: τ and error (kabc/kabc-sg), ARI-at-cap (acb)

In [4]:
rows = []
for algo in [a for a in ALGO_ORDER if a in main_df["algorithm"].unique()]:
    sub = main_df[main_df["algorithm"] == algo]
    rows.append({
        "algorithm": ALGO_LABELS[algo],
        "mean_tau": sub["tau"].mean(),
        "mean_error": sub["error"].mean(),
        "mean_ari": sub["ari"].mean(),
        "frac_stopped": sub["stopped"].mean(),
    })
fig2_table = pd.DataFrame(rows)
fig2_table

,algorithm,mean_tau,mean_error,mean_ari,frac_stopped
0,KABC,19672.00,0.00,1.000000,1.00
1,KABC-SG,69326.40,0.63,0.860876,0.71
2,ACB,1999900.76,1.00,0.000000,0.00
3,Unif-KKM,2000000.00,0.00,1.000000,1.00


## Figure 3: small-instance table — tas-fw vs kabc on the N=6 reduced instance

In [5]:
rows = []
for algo in [a for a in ["tas-fw", "kabc"] if a in reduced_df["algorithm"].unique()]:
    sub = reduced_df[reduced_df["algorithm"] == algo]
    rows.append({
        "algorithm": ALGO_LABELS[algo],
        "mean_tau": sub["tau"].mean(),
        "mean_error": sub["error"].mean(),
        "mean_ari": sub["ari"].mean(),
        "mean_wall_clock_s": sub["wall_clock_seconds"].mean(),
    })
fig3_table = pd.DataFrame(rows)
fig3_table

,algorithm,mean_tau,mean_error,mean_ari,mean_wall_clock_s
0,TaS-FW,4356.64,0.00,1.000000,39.638844
1,KABC,1852.20,0.02,0.995238,0.229660
